# Phase 5 — Fine-tuning ST-CDGM Stage 1 avec SRE (Option C)

## Stratégie (warm-start joint)
```
Checkpoint 15 epochs (A_dag parfait, Q_phys=0.9998)
         │
   Phase 5A (3 epochs)
   Freeze GNN+RCN+head  — SRE seul
   Loss = MSE(δ, HR − μ_causal.detach())
         │
   Phase 5B (7 epochs)
   Unfreeze GNN+RCN+head — A_dag reste gelé
   Loss = MSE(μ_total, HR) + λ_causal · MSE(μ_causal, HR)
   λ_causal : 1.0 → 0.3  (schedule linéaire)
         │
   Recalibration σ_data Stage 2  (1 passe, ~5 min)
```

## Invariants à préserver
| Invariant | Contrôle |
|---|---|
| A_dag gelé toute la durée | A_dag.requires_grad_(False) + assert drift<1e-6 |
| Q_phys évalué sur μ_causal seul | std(μ_causal) ne doit pas chuter |
| μ_causal_frac ≥ 0.3 en fin de Phase B | SRE ne doit pas dominer |
| δ ≈ 0 à l'init (SRE zero-init) | T2 passé en smoke test |

## Outputs
- `epoch_best_stage1_with_sre.pth` : encoder+RCN+head+SRE
- `sigma_data_sre.json` : nouvelle σ_data pour Stage 2

In [ ]:
# === Cell 1 : Bootstrap ===
import subprocess, shlex, os, sys
from pathlib import Path

REPO_DIR   = Path('/content/climate_data')
GIT_URL    = 'https://github.com/leonelkenfack/stcdgm.git'
GIT_BRANCH = 'four-node-causal'

if not (REPO_DIR / '.git').exists():
    subprocess.run(shlex.split(
        f'git clone --depth 200 -b {GIT_BRANCH} {GIT_URL} {REPO_DIR}'), check=True)
else:
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} fetch --depth=200 origin {GIT_BRANCH}'), check=True)
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} reset --hard origin/{GIT_BRANCH}'), check=True)

os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))

try:
    import torch_geometric; import cftime; import h5netcdf; import xbatcher; import diffusers
    from omegaconf import OmegaConf
except ImportError:
    EXTRA_DEPS = [
        'torch_geometric', 'omegaconf==2.3.0', 'hydra-core==1.3.2',
        'diffusers==0.36.0', 'einops', 'scipy', 'h5py', 'netCDF4',
        'xarray', 'dask', 'zarr', 'safetensors==0.7.0',
        'xbatcher', 'webdataset', 'cftime', 'h5netcdf',
    ]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + EXTRA_DEPS, check=True)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ModuleNotFoundError:
    print('[bootstrap] not on Colab — Drive mount skipped')

print(f'[bootstrap] cwd={os.getcwd()}  branch={GIT_BRANCH}')

In [ ]:
# === Cell 2 : Imports + constantes ===
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from contextlib import nullcontext
from pathlib import Path
from omegaconf import OmegaConf
from torch.optim import Adam

from st_cdgm.models.spatial_residual_encoder import SpatialResidualEncoder
from st_cdgm.training.stage1_paths import batch_lr_grid_last
from st_cdgm.training.two_stage import stage1_compute_loss

# --- Chemins Drive ---
DRIVE_ROOT     = Path('/content/drive/MyDrive/climate_data')
ORACLE_9N      = DRIVE_ROOT / 'oracle_9node' / 'seed_42'
CKPT_STAGE1_IN = ORACLE_9N / 'epoch_best_stage1.pth'
CKPT_SRE_OUT   = ORACLE_9N / 'epoch_best_stage1_with_sre.pth'
SIGMA_JSON     = ORACLE_9N / 'sigma_data_sre.json'

assert CKPT_STAGE1_IN.exists(), f'Checkpoint introuvable : {CKPT_STAGE1_IN}'
print(f'[Cell 2] Checkpoint source : {CKPT_STAGE1_IN}')
print(f'[Cell 2] Checkpoint sortie : {CKPT_SRE_OUT}')

# --- Phase 5A : SRE seul (backbone gelé) ---
PHASE_A_EPOCHS = 3
PHASE_A_LR     = 3e-4

# --- Phase 5B : Joint fine-tuning (tout sauf A_dag) ---
PHASE_B_EPOCHS          = 7
PHASE_B_LR_BACKBONE     = 3e-5   # LR conservateur pour préserver le DAG
PHASE_B_LR_SRE          = 1e-4   # SRE peut apprendre plus vite
LAMBDA_CAUSAL_START     = 1.0    # identifiabilité causale au départ
LAMBDA_CAUSAL_END       = 0.3    # relâchement progressif
GRADIENT_CLIPPING       = 1.0

# --- SRE config (identique au smoke test) ---
SRE_BASE_CH  = 32
SRE_D_COND   = 128   # = hidden_dim RCN
H_HR, W_HR   = 172, 179

# --- Critères de succès ---
# Phase 5A : loss résidu diminue
# Phase 5B : std(μ_total) > std(μ_causal_baseline) × 1.1
#            RMSE(μ_causal) ne chute pas de plus de 10%
#            μ_causal_frac ≥ 0.3 en fin de run
SUCCESS_STD_GAIN   = 1.10   # std(μ_total) doit gagner 10%
SUCCESS_RMSE_MAX   = 1.10   # RMSE(μ_causal) ne monte pas > 10%
SUCCESS_FRAC_MIN   = 0.30   # SRE ne doit pas dominer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[Cell 2] DEVICE={DEVICE}  Phase A: {PHASE_A_EPOCHS}ep×LR={PHASE_A_LR}  Phase B: {PHASE_B_EPOCHS}ep×LR_bb={PHASE_B_LR_BACKBONE}')

In [ ]:
# === Cell 3 : Config + pipeline + dataloaders (9-node) ===
from torch.utils.data import DataLoader as _DataLoader, IterableDataset
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from path_c_plus.scripts.option_c_helpers import PATHCPLUS_HYPERPARAM_OVERRIDES
from path_c_plus.scripts.gpu_detect import detect_gpu_profile, print_profile_banner

# --- Config ---
CONFIG = OmegaConf.load('config/training_config.yaml')
_corrdiff = OmegaConf.load('config/training_config_corrdiff_normal.yaml')
CONFIG = OmegaConf.merge(CONFIG, _corrdiff)

EXTENDED_9NODE = True
GPU_PROFILE = detect_gpu_profile()
print_profile_banner(GPU_PROFILE)
CONFIG.training.batch_size = GPU_PROFILE['batch_size']
CONFIG.training.use_amp    = GPU_PROFILE['use_amp']
CONFIG.training.num_workers = GPU_PROFILE['num_workers']

ts_cfg = CONFIG.two_stage
ts_cfg.stage1['lambda_dag_prior'] = PATHCPLUS_HYPERPARAM_OVERRIDES['lambda_dag_prior']
ts_cfg.stage1['g_phys_alpha']     = PATHCPLUS_HYPERPARAM_OVERRIDES['g_phys_alpha']

# 9-node : inject humid metapaths
OmegaConf.set_struct(CONFIG, False)
_existing_mp = {m.name for m in CONFIG.encoder.metapaths}
for _m in [
    {'name': 'Q850', 'src': 'Q850', 'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
    {'name': 'W500', 'src': 'W500', 'relation': 'causes', 'target': 'GP500', 'pool': 'mean'},
    {'name': 'IVT',  'src': 'IVT',  'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
]:
    if _m['name'] not in _existing_mp:
        CONFIG.encoder.metapaths.append(OmegaConf.create(_m))
print(f'[Cell 3] metapaths -> {[m.name for m in CONFIG.encoder.metapaths]}')

# --- K9 split ---
K9_DATES = {
    'train':   ['1980-01-01', '2009-12-31'],
    'val':     ['2010-01-01', '2011-12-31'],
    'test':    ['2012-01-01', '2013-12-31'],
    'holdout': ['2014-01-01', '2014-12-31'],
}

_ON_COLAB  = 'google.colab' in sys.modules or Path('/content').exists()
DATA_ROOT  = Path('/content/drive/MyDrive/climate_data/data') if _ON_COLAB else Path('data/raw')
LR_PATH    = str(DATA_ROOT / 'train' / 'predictor_ACCESS-CM2_hist.nc')
HR_PATH    = str(DATA_ROOT / 'train' / 'pr_ACCESS-CM2_hist.nc')
_static_p  = DATA_ROOT / 'static_predictors' / 'ERA5_eval_ccam_12km.198110_NZ_Invariant.nc'
_mean_p    = DATA_ROOT / 'train' / 'means_ACCESS-CM2.nc'
_std_p     = DATA_ROOT / 'train' / 'stds_ACCESS-CM2.nc'
STATIC_PATH = str(_static_p) if _static_p.exists() else None
MEAN_PATH   = str(_mean_p)   if _mean_p.exists()   else None
STD_PATH    = str(_std_p)    if _std_p.exists()    else None

SEQ_LEN            = int(CONFIG.data.seq_len)
BASELINE_STRATEGY  = str(CONFIG.data.baseline_strategy)
BASELINE_FACTOR    = int(CONFIG.data.baseline_factor)
NORMALIZE          = bool(CONFIG.data.normalize)
PRECIPITATION_DELTA = float(CONFIG.data.precipitation_delta)
NAN_FILL_STRATEGY  = str(CONFIG.data.nan_fill_strategy)
_default_lr = ['q_500', 'q_850', 'u_500', 'u_850', 'v_500', 'v_850', 't_500', 't_850']
LR_VARIABLES  = list(CONFIG.data.lr_variables)  if CONFIG.data.get('lr_variables')  else _default_lr
HR_VARIABLES  = list(CONFIG.data.hr_variables)  if CONFIG.data.get('hr_variables')  else ['pr']
STATIC_VARIABLES = list(CONFIG.data.static_variables) if CONFIG.data.get('static_variables') else []

pipeline = NetCDFDataPipeline(
    lr_path=LR_PATH, hr_path=HR_PATH, static_path=STATIC_PATH,
    seq_len=SEQ_LEN, baseline_strategy=BASELINE_STRATEGY,
    baseline_factor=BASELINE_FACTOR, normalize=NORMALIZE,
    nan_fill_strategy=NAN_FILL_STRATEGY,
    precipitation_delta=PRECIPITATION_DELTA,
    lr_variables=LR_VARIABLES, hr_variables=HR_VARIABLES,
    static_variables=STATIC_VARIABLES,
    means_path=MEAN_PATH, stds_path=STD_PATH,
    train_start_date=K9_DATES['train'][0], train_end_date=K9_DATES['train'][1],
    val_start_date=K9_DATES['val'][0],     val_end_date=K9_DATES['val'][1],
    test_start_date=K9_DATES['test'][0],   test_end_date=K9_DATES['test'][1],
    temporal_holdout_start_date=K9_DATES['holdout'][0],
    temporal_holdout_end_date=K9_DATES['holdout'][1],
)
print('[Cell 3] Pipeline ready')

train_dataset = pipeline.build_sequence_dataset(split='train', seq_len=SEQ_LEN,
                                                 stride=int(CONFIG.data.stride), as_torch=True)
val_dataset   = pipeline.build_sequence_dataset(split='val',   seq_len=SEQ_LEN,
                                                 stride=int(CONFIG.data.stride), as_torch=True)

BATCH_SIZE  = int(CONFIG.training.batch_size)
NUM_WORKERS = int(CONFIG.training.num_workers)
PIN_MEMORY  = bool(torch.cuda.is_available())
_loader_kw  = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                   pin_memory=PIN_MEMORY, collate_fn=lambda x: x)
if NUM_WORKERS > 0:
    _loader_kw.update(persistent_workers=True, prefetch_factor=2)

train_dataloader = _DataLoader(train_dataset,
                                shuffle=not isinstance(train_dataset, IterableDataset),
                                **_loader_kw)
val_dataloader   = _DataLoader(val_dataset, shuffle=False, **_loader_kw)

# --- Builder (9-node) ---
lr_shape = tuple(CONFIG.graph.lr_shape)
hr_shape = tuple(CONFIG.graph.hr_shape)
builder  = HeteroGraphBuilder(
    lr_shape=lr_shape, hr_shape=hr_shape,
    static_dataset=pipeline.get_static_dataset(),
    include_mid_layer=CONFIG.graph.include_mid_layer,
    extended_9node=EXTENDED_9NODE,
)
print(f'[Cell 3] Builder  lr_shape={lr_shape}  hr_shape={hr_shape}  dyn={builder.dynamic_node_types}')

# --- 9-node convert_sample_to_batch (inclut lr_grid pour SRE) ---
_LR_VARS = list(CONFIG.data.lr_variables)
_VI = {v: i for i, v in enumerate(_LR_VARS)}
_Q_IDX = [_VI[v] for v in ('q_850', 'q_500', 'q_250') if v in _VI]
_W_IDX = [_VI[v] for v in ('w_850', 'w_500', 'w_250') if v in _VI]
_IVT_LEVELS = [lev for lev in ('850', '500', '250')
               if f'q_{lev}' in _VI and f'u_{lev}' in _VI and f'v_{lev}' in _VI]

def _compute_ivt_nodes(lr0):
    acc = None
    for lev in _IVT_LEVELS:
        q = lr0[:, _VI[f'q_{lev}']]
        u = lr0[:, _VI[f'u_{lev}']]
        v = lr0[:, _VI[f'v_{lev}']]
        term = q * torch.sqrt(u * u + v * v + 1e-12)
        acc = term if acc is None else acc + term
    if acc is None:
        acc = lr0[:, 0:1] * 0.0
    return acc / (len(_IVT_LEVELS) + 1e-8)

def convert_sample_to_batch(sample, builder, device):
    """Converts one dataset sample to a batch dict.
    Preserves lr_grid (raw [T,C,H,W]) alongside lr (node tensor) for SRE.
    """
    lr_seq = sample['lr']                    # [T, C, H_lr, W_lr]  raw grid
    seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)   # [T, N, C]  node tensor
    lr0 = lr_nodes_steps[0]
    if EXTENDED_9NODE:
        _ivt = _compute_ivt_nodes(lr0)
        dynamic_features = {}
        for nt in builder.dynamic_node_types:
            if nt == 'Q850':   dynamic_features[nt] = lr0[:, _Q_IDX] if _Q_IDX else lr0
            elif nt == 'W500': dynamic_features[nt] = lr0[:, _W_IDX] if _W_IDX else lr0
            elif nt == 'IVT':  dynamic_features[nt] = _ivt
            else:              dynamic_features[nt] = lr0
    else:
        dynamic_features = {nt: lr0 for nt in builder.dynamic_node_types}
    hetero = builder.prepare_step_data(dynamic_features).to(device)
    return {
        'lr':      lr_tensor,    # [T, N, C]  pour RCN
        'lr_grid': lr_seq,       # [T, C, H, W]  pour SRE via batch_lr_grid_last
        'residual': sample['residual'],
        'baseline': sample.get('baseline'),
        'hetero':   hetero,
        'time':     sample.get('time'),
    }

def iterate_batches(dataloader, builder, device):
    for batch_list in dataloader:
        if not isinstance(batch_list, list):
            batch_list = [batch_list]
        yield [convert_sample_to_batch(s, builder, device) for s in batch_list]

_probe = next(iter(train_dataset))
C_LR = _probe['lr'].shape[1]   # channels LR (15 avec variables statiques)
_n_train = len(train_dataset) if hasattr(train_dataset, '__len__') else '?'
_n_val   = len(val_dataset)   if hasattr(val_dataset,   '__len__') else '?'
print(f'[Cell 3] C_LR={C_LR}  lr_shape={lr_shape}  hr_shape={hr_shape}')
print(f'[Cell 3] train={_n_train} samples  val={_n_val} samples')

In [ ]:
# === Cell 4 : Build stack 9-node + charger epoch_best_stage1.pth ===
from st_cdgm.models.intelligible_encoder import (
    IntelligibleVariableEncoder, IntelligibleVariableConfig,
)
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.regression_head import GraphToGridDecoder

def _build_encoder(builder, CONFIG, device):
    allowed = set(builder.dynamic_node_types) | set(builder.static_node_types)
    cfgs = []
    for _mp in CONFIG.encoder.metapaths:
        if _mp.src in allowed and _mp.target in allowed:
            cfgs.append(IntelligibleVariableConfig(
                name=_mp.name,
                meta_path=(_mp.src, _mp.relation, _mp.target),
                pool=_mp.get('pool', 'mean'),
            ))
    if pipeline.get_static_dataset() is not None:
        cfgs.append(IntelligibleVariableConfig(
            name='static', meta_path=('SP_HR', 'causes', 'GP850'), pool='mean',
        ))
    enc = IntelligibleVariableEncoder(
        configs=cfgs,
        hidden_dim=int(CONFIG.encoder.hidden_dim),
        conditioning_dim=int(CONFIG.encoder.conditioning_dim),
    ).to(device)
    return enc, len(cfgs)

torch.manual_seed(SEED); np.random.seed(SEED)
encoder, num_vars = _build_encoder(builder, CONFIG, DEVICE)

_probe_b = next(iter(train_dataset))
_lr_nodes = builder.lr_grid_to_nodes(_probe_b['lr'][0])
RCN_DRIVER_DIM = _lr_nodes.shape[-1]

rcn_cell = RCNCell(
    num_vars=num_vars,
    hidden_dim=int(CONFIG.rcn.hidden_dim),
    driver_dim=RCN_DRIVER_DIM,
    reconstruction_dim=RCN_DRIVER_DIM,
    dropout=float(CONFIG.rcn.dropout),
).to(DEVICE)
rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get('detach_interval'))

rh_cfg = CONFIG.two_stage.regression_head
regression_head = GraphToGridDecoder(
    d_model=int(rh_cfg.d_model),
    hr_h=H_HR, hr_w=W_HR,
    intermediate_h=int(rh_cfg.intermediate_h),
    intermediate_w=int(rh_cfg.intermediate_w),
    n_heads=int(rh_cfg.n_heads),
    refine_channels=int(rh_cfg.refine_channels),
    output_channels=1,
).to(DEVICE)

print('[Cell 4] Stack instancié. Chargement checkpoint...')

def _load_sd(module, key, ck):
    sd = ck.get(key)
    if sd is None:
        print(f'  WARN {key} absent du checkpoint')
        return
    if isinstance(sd, dict) and any('_orig_mod' in k for k in sd):
        sd = {k.replace('_orig_mod.', ''): v for k, v in sd.items()}
    module.load_state_dict(sd, strict=True)

ck = torch.load(CKPT_STAGE1_IN, map_location=DEVICE, weights_only=False)
_load_sd(encoder,         'encoder_state_dict',         ck)
_load_sd(rcn_cell,        'rcn_cell_state_dict',         ck)
_load_sd(regression_head, 'regression_head_state_dict',  ck)

_n_enc = sum(p.numel() for p in encoder.parameters())
_n_rcn = sum(p.numel() for p in rcn_cell.parameters())
_n_rh  = sum(p.numel() for p in regression_head.parameters())
print(f'[Cell 4] Checkpoint chargé  epoch={ck.get("epoch","?")}  '
      f'enc={_n_enc:,}  rcn={_n_rcn:,}  head={_n_rh:,}')

# Geler A_dag définitivement (skeleton_F1=1.0, Q_phys=0.9998 → ne pas toucher)
_rcn_core = rcn_cell
if hasattr(_rcn_core, '_orig_mod'):
    _rcn_core = _rcn_core._orig_mod
if hasattr(_rcn_core, 'A_dag'):
    _rcn_core.A_dag.requires_grad_(False)
    _A_dag_ref = _rcn_core.A_dag.detach().clone()
    print(f'[Cell 4] A_dag gelé  shape={tuple(_rcn_core.A_dag.shape)}  '
          f'norm={_A_dag_ref.norm():.4f}  asym={(_A_dag_ref - _A_dag_ref.T).abs().mean():.4f}')
else:
    _A_dag_ref = None
    print('[Cell 4] WARN : A_dag introuvable')

# Phase 5A : geler tout le backbone (SRE seul entraînable)
def _freeze_backbone():
    for p in encoder.parameters():         p.requires_grad_(False)
    for p in rcn_cell.parameters():        p.requires_grad_(False)
    for p in regression_head.parameters(): p.requires_grad_(False)
    if _A_dag_ref is not None:
        _rcn_core.A_dag.requires_grad_(False)  # redondant mais explicite
    print('[freeze_backbone] encoder+RCN+head gelés.')

def _unfreeze_backbone():
    for p in encoder.parameters():         p.requires_grad_(True)
    for p in rcn_cell.parameters():        p.requires_grad_(True)
    for p in regression_head.parameters(): p.requires_grad_(True)
    if _A_dag_ref is not None:
        _rcn_core.A_dag.requires_grad_(False)   # A_dag reste gelé
    print('[unfreeze_backbone] encoder+RCN+head dégelés. A_dag reste gelé.')

_freeze_backbone()
print('[Cell 4] Backbone prêt pour Phase 5A.')

In [ ]:
# === Cell 5 : Instantiation SRE ===
sre = SpatialResidualEncoder(
    in_channels=C_LR,
    d_cond=SRE_D_COND,
    base_ch=SRE_BASE_CH,
    hr_h=H_HR,
    hr_w=W_HR,
    out_gain=0.1,
).to(DEVICE)

_n_sre = sre.num_params()
print(f'[Cell 5] SRE instancié : {_n_sre:,} params  (contrainte < 500k)')
assert _n_sre < 500_000, f'SRE trop grand ({_n_sre:,} > 500k) !'

# Vérifier zero-init
with torch.no_grad():
    _lr_test  = torch.zeros(1, C_LR, *lr_shape, device=DEVICE)
    _HT_test  = torch.zeros(1, num_vars, 1, SRE_D_COND, device=DEVICE)
    _delta_0  = sre(_lr_test, _HT_test).abs().max().item()
assert _delta_0 < 1e-4, f'Zero-init cassé : |δ|_max={_delta_0:.2e}'
print(f'[Cell 5] Zero-init OK  |δ|_max={_delta_0:.2e}')
print(f'[Cell 5] SRE prêt.')

# Optimizer Phase 5A : SRE seul
optimizer_a = Adam(sre.parameters(), lr=PHASE_A_LR)
print(f'[Cell 5] Optimizer Phase A : Adam(lr={PHASE_A_LR}, params_sre={_n_sre:,})')

In [ ]:
# === Cell 6 : Probe baseline AVANT entraînement ===

def _forward_one(micro, device, with_grad=False):
    """Forward pass d'un micro-batch. Retourne mu_causal, delta, mu_total, H_T."""
    ctx = torch.enable_grad() if with_grad else torch.no_grad()
    with ctx:
        lr_data = micro['lr'].to(device)    # [T, N, C]
        h_init  = encoder.init_state(micro['hetero']).to(device)  # [q, N, d]
        drivers = [lr_data[t] for t in range(lr_data.shape[0])]
        seq_out = rcn_runner.run(h_init, drivers, reconstruction_sources=None)
        H_T     = seq_out.states[-1]        # [q, N, d]  — single sample, no batch dim
        mu_causal = regression_head(H_T)    # [1, 1, H_HR, W_HR] ou [1, H, W]
        if mu_causal.dim() == 3:
            mu_causal = mu_causal.unsqueeze(0)
        # SRE attend [B, q, N, d] — unsqueeze(0) ajoute la dim batch
        lr_grid = batch_lr_grid_last(micro, builder=builder, device=device)  # [1, C, H_lr, W_lr]
        lr_safe = torch.nan_to_num(lr_grid, nan=0.0)
        H_T_b   = H_T.unsqueeze(0)         # [1, q, N, d]  batch dim pour SRE
        delta   = sre(lr_safe, H_T_b)      # [1, 1, H_HR, W_HR]
        mu_tot  = mu_causal + delta
    return mu_causal, delta, mu_tot, H_T

@torch.no_grad()
def _val_probe(label=''):
    encoder.eval(); rcn_cell.eval(); regression_head.eval(); sre.eval()
    std_c_list, std_t_list, err_c, err_t, delta_stds, frac_list = [], [], [], [], [], []
    for batch_list in iterate_batches(val_dataloader, builder, DEVICE):
        for micro in batch_list:
            tgt = micro['residual'][-1].to(DEVICE)
            if tgt.dim() == 3: tgt = tgt.unsqueeze(0)
            valid = torch.isfinite(tgt)
            mu_c, delta, mu_t, _ = _forward_one(micro, DEVICE)
            std_c_list.append(mu_c[valid].std().item())
            std_t_list.append(mu_t[valid].std().item())
            delta_stds.append(delta.std().item())
            tgt_v = tgt[valid]
            err_c.append((mu_c[valid] - tgt_v).pow(2).mean().item())
            err_t.append((mu_t[valid] - tgt_v).pow(2).mean().item())
            # μ_causal_frac
            num = mu_c.abs().mean().item()
            den = mu_c.abs().mean().item() + delta.abs().mean().item() + 1e-12
            frac_list.append(num / den)

    std_c  = float(np.mean(std_c_list))
    std_t  = float(np.mean(std_t_list))
    rmse_c = float(np.mean(err_c)) ** 0.5
    rmse_t = float(np.mean(err_t)) ** 0.5
    d_std  = float(np.mean(delta_stds))
    frac   = float(np.mean(frac_list))
    print(f'[probe {label}]')
    print(f'  std(μ_causal)={std_c:.4f}  std(μ_total)={std_t:.4f}')
    print(f'  RMSE_causal={rmse_c:.5f}  RMSE_total={rmse_t:.5f}')
    print(f'  mean(δ_std)={d_std:.5f}  μ_causal_frac={frac:.3f}')
    return dict(std_c=std_c, std_t=std_t, rmse_c=rmse_c, rmse_t=rmse_t,
                delta_std=d_std, frac=frac)

print('=== Probe BASELINE (SRE zero-init, δ≈0) ===')
metrics_baseline = _val_probe('BASELINE')
if _A_dag_ref is not None:
    print(f'  A_dag norm={_A_dag_ref.norm():.4f}')
print(f'  → std(μ_total) cible : > {metrics_baseline["std_t"]*SUCCESS_STD_GAIN:.4f}')
print(f'  → RMSE_causal limite : < {metrics_baseline["rmse_c"]*SUCCESS_RMSE_MAX:.5f}')

In [ ]:
# === Cell 7 : Phase 5A — SRE seul (backbone gelé, 3 epochs) ===
# Loss = MSE(δ, HR − μ_causal.detach()) sur pixels valides
# Objectif : SRE apprend le résidu spatial rapidement avant le joint training

def _autocast(device, use_amp):
    if use_amp and device.type == 'cuda':
        return torch.amp.autocast(device_type='cuda', dtype=torch.float16)
    return nullcontext()

USE_AMP = bool(CONFIG.training.use_amp)

def _phase_a_epoch(ep_idx):
    sre.train()
    encoder.eval(); rcn_cell.eval(); regression_head.eval()
    scaler = torch.amp.GradScaler(enabled=(USE_AMP and DEVICE.type == 'cuda'))
    total_loss = 0.0; n_batches = 0

    for batch_idx, batch_list in enumerate(iterate_batches(train_dataloader, builder, DEVICE)):
        optimizer_a.zero_grad(set_to_none=True)
        step_loss = 0.0

        for micro in batch_list:
            tgt = micro['residual'][-1].to(DEVICE)
            if tgt.dim() == 3: tgt = tgt.unsqueeze(0)
            valid = torch.isfinite(tgt)
            tgt_clean = torch.nan_to_num(tgt, nan=0.0)

            # Backbone gelé → no_grad
            with torch.no_grad():
                lr_data = micro['lr'].to(DEVICE)
                h_init  = encoder.init_state(micro['hetero']).to(DEVICE)
                drivers = [lr_data[t] for t in range(lr_data.shape[0])]
                seq_out = rcn_runner.run(h_init, drivers, reconstruction_sources=None)
                H_T     = seq_out.states[-1]         # [q, N, d]
                mu_c    = regression_head(H_T)
                if mu_c.dim() == 3: mu_c = mu_c.unsqueeze(0)

            # SRE avec grad
            with _autocast(DEVICE, USE_AMP):
                lr_grid  = batch_lr_grid_last(micro, builder=builder, device=DEVICE)
                lr_safe  = torch.nan_to_num(lr_grid, nan=0.0)
                H_T_b    = H_T.detach().unsqueeze(0)  # batch dim pour SRE, detached
                delta    = sre(lr_safe, H_T_b)
                resid_tgt = torch.nan_to_num(tgt_clean - mu_c.detach(), nan=0.0)
                loss = F.mse_loss(delta[valid], resid_tgt[valid])
                loss = loss / max(len(batch_list), 1)

            if scaler.is_enabled(): scaler.scale(loss).backward()
            else:                   loss.backward()
            step_loss += loss.item()

        if GRADIENT_CLIPPING > 0:
            if scaler.is_enabled(): scaler.unscale_(optimizer_a)
            torch.nn.utils.clip_grad_norm_(sre.parameters(), GRADIENT_CLIPPING)

        if scaler.is_enabled(): scaler.step(optimizer_a); scaler.update()
        else:                   optimizer_a.step()

        total_loss += step_loss; n_batches += 1
        if batch_idx == 0 or (batch_idx + 1) % 30 == 0:
            print(f'  [5A ep{ep_idx}] batch {batch_idx+1} | loss={step_loss:.5f}  '
                  f'out_gain={sre.out_gain.item():.4f}', flush=True)

    return total_loss / max(1, n_batches)

print('=' * 60)
print(f'PHASE 5A : SRE seul  ({PHASE_A_EPOCHS} epochs, LR={PHASE_A_LR})')
print('=' * 60)
phase_a_losses = []
for ep in range(1, PHASE_A_EPOCHS + 1):
    print(f'\n--- Phase 5A epoch {ep}/{PHASE_A_EPOCHS} ---')
    loss_a = _phase_a_epoch(ep)
    print(f'  mean loss={loss_a:.5f}  out_gain={sre.out_gain.item():.4f}')
    phase_a_losses.append(loss_a)

    # Vérifier A_dag intact
    if _A_dag_ref is not None:
        drift = (_rcn_core.A_dag.detach() - _A_dag_ref).abs().max().item()
        assert drift < 1e-6, f'A_dag a bougé : drift={drift:.2e}'
        print(f'  A_dag drift={drift:.1e}  OK')

print('\n=== Probe après Phase 5A ===')
metrics_after_a = _val_probe('AFTER-5A')
print(f'  out_gain final : {sre.out_gain.item():.4f}')

In [ ]:
# === Cell 8 : Phase 5B — Joint fine-tuning (backbone + SRE, A_dag gelé) ===
# Loss = MSE(μ_total, HR) + λ_causal · MSE(μ_causal, HR)
# λ_causal : schedule linéaire 1.0 → 0.3 sur PHASE_B_EPOCHS

# Dégeler le backbone (A_dag reste gelé)
_unfreeze_backbone()

# Optimizer joint : LR différencié backbone (conservateur) / SRE
optimizer_b = Adam([
    {'params': list(encoder.parameters()),         'lr': PHASE_B_LR_BACKBONE},
    {'params': list(rcn_cell.parameters()),         'lr': PHASE_B_LR_BACKBONE},
    {'params': list(regression_head.parameters()),  'lr': PHASE_B_LR_BACKBONE},
    {'params': list(sre.parameters()),              'lr': PHASE_B_LR_SRE},
])

def _lambda_causal_schedule(ep_b):
    """Linéaire : 1.0 à l'ep 1, 0.3 à l'ep PHASE_B_EPOCHS."""
    frac = (ep_b - 1) / max(1, PHASE_B_EPOCHS - 1)
    return LAMBDA_CAUSAL_START * (1 - frac) + LAMBDA_CAUSAL_END * frac

def _phase_b_epoch(ep_idx, lambda_causal):
    sre.train(); encoder.train(); rcn_cell.train(); regression_head.train()
    scaler = torch.amp.GradScaler(enabled=(USE_AMP and DEVICE.type == 'cuda'))
    total_main = 0.0; total_causal = 0.0; n_batches = 0

    for batch_idx, batch_list in enumerate(iterate_batches(train_dataloader, builder, DEVICE)):
        optimizer_b.zero_grad(set_to_none=True)
        step_main = 0.0; step_causal = 0.0

        for micro in batch_list:
            tgt = micro['residual'][-1].to(DEVICE)
            if tgt.dim() == 3: tgt = tgt.unsqueeze(0)
            valid   = torch.isfinite(tgt)
            tgt_cln = torch.nan_to_num(tgt, nan=0.0)

            with _autocast(DEVICE, USE_AMP):
                # Causal forward pass (avec grad)
                lr_data = micro['lr'].to(DEVICE)
                h_init  = encoder.init_state(micro['hetero']).to(DEVICE)
                drivers = [lr_data[t] for t in range(lr_data.shape[0])]
                seq_out = rcn_runner.run(h_init, drivers, reconstruction_sources=None)
                H_T     = seq_out.states[-1]           # [q, N, d]
                mu_c    = regression_head(H_T)
                if mu_c.dim() == 3: mu_c = mu_c.unsqueeze(0)

                # SRE forward pass (H_T partagé, gradient passe dans les deux)
                lr_grid  = batch_lr_grid_last(micro, builder=builder, device=DEVICE)
                lr_safe  = torch.nan_to_num(lr_grid, nan=0.0)
                H_T_b    = H_T.unsqueeze(0)            # [1, q, N, d]
                delta    = sre(lr_safe, H_T_b)
                mu_tot   = mu_c + delta

                # Loss combinée
                loss_main   = F.mse_loss(mu_tot[valid], tgt_cln[valid])
                loss_causal = F.mse_loss(mu_c[valid],   tgt_cln[valid])
                loss = (loss_main + lambda_causal * loss_causal) / max(len(batch_list), 1)

            if scaler.is_enabled(): scaler.scale(loss).backward()
            else:                   loss.backward()
            step_main   += loss_main.item()
            step_causal += loss_causal.item()

        # Gradient clipping
        all_params = (list(encoder.parameters()) + list(rcn_cell.parameters()) +
                      list(regression_head.parameters()) + list(sre.parameters()))
        if GRADIENT_CLIPPING > 0:
            if scaler.is_enabled(): scaler.unscale_(optimizer_b)
            torch.nn.utils.clip_grad_norm_(all_params, GRADIENT_CLIPPING)

        if scaler.is_enabled(): scaler.step(optimizer_b); scaler.update()
        else:                   optimizer_b.step()

        # A_dag gelé → re-forcer (sécurité)
        if _A_dag_ref is not None:
            _rcn_core.A_dag.requires_grad_(False)

        total_main += step_main; total_causal += step_causal; n_batches += 1
        if batch_idx == 0 or (batch_idx + 1) % 30 == 0:
            print(f'  [5B ep{ep_idx}] batch {batch_idx+1} | '
                  f'main={step_main:.5f}  causal={step_causal:.5f}  '
                  f'λ_c={lambda_causal:.2f}', flush=True)

    return dict(loss_main=total_main/max(1,n_batches),
                loss_causal=total_causal/max(1,n_batches),
                lambda_causal=lambda_causal)

print('=' * 60)
print(f'PHASE 5B : Joint fine-tuning  ({PHASE_B_EPOCHS} epochs)')
print(f'  LR backbone={PHASE_B_LR_BACKBONE}  LR SRE={PHASE_B_LR_SRE}')
print(f'  λ_causal : {LAMBDA_CAUSAL_START} → {LAMBDA_CAUSAL_END}')
print('=' * 60)

phase_b_metrics = []
best_rmse_total = float('inf')

for ep_b in range(1, PHASE_B_EPOCHS + 1):
    lc = _lambda_causal_schedule(ep_b)
    print(f'\n--- Phase 5B epoch {ep_b}/{PHASE_B_EPOCHS}  λ_causal={lc:.3f} ---')
    m = _phase_b_epoch(ep_b, lc)
    print(f'  loss_main={m["loss_main"]:.5f}  loss_causal={m["loss_causal"]:.5f}')
    print(f'  out_gain={sre.out_gain.item():.4f}')

    # Vérification A_dag drift
    if _A_dag_ref is not None:
        drift = (_rcn_core.A_dag.detach() - _A_dag_ref).abs().max().item()
        print(f'  A_dag drift={drift:.1e}  (doit être 0)')
        assert drift < 1e-5, f'A_dag a drifté de {drift:.2e} !'

    # Probe val
    probe = _val_probe(f'5B-ep{ep_b}')
    m.update(probe)
    phase_b_metrics.append(m)

    # Sauvegarde checkpoint à chaque epoch (keep best RMSE_total)
    _ck = {
        'schema_version': 2,
        'phase_b_epoch': ep_b,
        'total_ft_epochs': PHASE_A_EPOCHS + ep_b,
        'encoder_state_dict':         encoder.state_dict(),
        'rcn_cell_state_dict':         rcn_cell.state_dict(),
        'regression_head_state_dict':  regression_head.state_dict(),
        'sre_state_dict':              sre.state_dict(),
        'optimizer_b_state_dict':      optimizer_b.state_dict(),
        'sre_config': {
            'in_channels': C_LR, 'd_cond': SRE_D_COND,
            'base_ch': SRE_BASE_CH, 'hr_h': H_HR, 'hr_w': W_HR,
        },
        'metrics_baseline': metrics_baseline,
        f'metrics_5b_ep{ep_b}': probe,
        'phase_a_losses': phase_a_losses,
        'lambda_causal': lc,
    }
    CKPT_SRE_OUT.parent.mkdir(parents=True, exist_ok=True)
    torch.save(_ck, CKPT_SRE_OUT)
    print(f'  Sauvegardé → {CKPT_SRE_OUT}')

    if probe['rmse_t'] < best_rmse_total:
        best_rmse_total = probe['rmse_t']
        _best = ORACLE_9N / 'epoch_best_stage1_with_sre_best.pth'
        torch.save(_ck, _best)
        print(f'  ★ Nouveau best RMSE_total={best_rmse_total:.5f} → {_best}')

print('\n[Cell 8] Phase 5B terminée.')

In [ ]:
# === Cell 9 : Recalibration σ_data Stage 2 ===
# Stage 2 EDM Karras est conditionné sur μ_HR. σ_data = std(HR − μ_HR).
# Avec SRE, μ_HR_total est différent de μ_HR_causal → recalibrer σ_data.
# On ne réentraîne PAS Stage 2 — juste σ_data change dans la config.

@torch.no_grad()
def calibrate_sigma_data():
    encoder.eval(); rcn_cell.eval(); regression_head.eval(); sre.eval()
    deltas = []
    for batch_list in iterate_batches(val_dataloader, builder, DEVICE):
        for micro in batch_list:
            tgt = micro['residual'][-1].to(DEVICE)
            if tgt.dim() == 3: tgt = tgt.unsqueeze(0)
            valid = torch.isfinite(tgt)
            mu_c, delta, mu_tot, _ = _forward_one(micro, DEVICE)
            diff = (tgt - mu_tot)[valid]
            diff = diff[torch.isfinite(diff)]
            if diff.numel() > 0:
                stride = max(1, diff.numel() // 512)
                deltas.append(diff[::stride].cpu())
    all_d = torch.cat(deltas)
    sigma_new  = all_d.std().item()
    mean_new   = all_d.mean().item()
    sigma_old  = 0.18   # valeur originale calibrée sur μ_HR_causal
    print(f'\n=== Recalibration σ_data Stage 2 ===')
    print(f'  σ_data ancienne (μ_causal) : {sigma_old:.5f}')
    print(f'  σ_data nouvelle (μ_total)  : {sigma_new:.5f}')
    print(f'  mean résidu               : {mean_new:.5f}')
    print(f'  n_pixels_eval             : {len(all_d):,}')
    if sigma_new > sigma_old * 1.1 or sigma_new < sigma_old * 0.9:
        print(f'  ⚠ Changement > 10% : mettre à jour sigma_data dans la config Stage 2.')
    else:
        print(f'  ✓ Changement < 10% : Stage 2 reste compatible sans réentraînement.')
    return {'sigma_data_old': sigma_old, 'sigma_data_new': sigma_new,
            'mean_residual': mean_new, 'n_pixels': int(len(all_d))}

sigma_result = calibrate_sigma_data()
SIGMA_JSON.write_text(json.dumps(sigma_result, indent=2))
print(f'Sauvegardé → {SIGMA_JSON}')

In [ ]:
# === Cell 10 : Verdict ===
import json

print('=' * 70)
print('PHASE 5 — VERDICT FINAL')
print('=' * 70)

# Comparaison baseline vs final
final_probe = phase_b_metrics[-1]  # dernière epoch Phase B
std_c_b  = metrics_baseline['std_c']
std_t_b  = metrics_baseline['std_t']
rmse_c_b = metrics_baseline['rmse_c']
frac_b   = metrics_baseline['frac']

std_c_f  = final_probe['std_c']
std_t_f  = final_probe['std_t']
rmse_c_f = final_probe['rmse_c']
frac_f   = final_probe['frac']

print(f'\n{"Métrique":<30} {"BASELINE":>12} {"FINAL":>12} {"Δ":>10}')
print('-' * 66)
print(f'{"std(μ_causal)":<30} {std_c_b:>12.4f} {std_c_f:>12.4f} {std_c_f-std_c_b:>+10.4f}')
print(f'{"std(μ_total)":<30} {std_t_b:>12.4f} {std_t_f:>12.4f} {std_t_f-std_t_b:>+10.4f}')
print(f'{"RMSE(μ_causal)":<30} {rmse_c_b:>12.5f} {rmse_c_f:>12.5f} {rmse_c_f-rmse_c_b:>+10.5f}')
print(f'{"μ_causal_frac":<30} {frac_b:>12.3f} {frac_f:>12.3f} {frac_f-frac_b:>+10.3f}')
print(f'{"σ_data Stage 2":<30} {sigma_result["sigma_data_old"]:>12.5f} {sigma_result["sigma_data_new"]:>12.5f}')
print('-' * 66)

# Critères de succès
OK_STD   = std_t_f  > std_t_b  * SUCCESS_STD_GAIN
OK_RMSE  = rmse_c_f < rmse_c_b * SUCCESS_RMSE_MAX
OK_FRAC  = frac_f   >= SUCCESS_FRAC_MIN
OK_DAG   = True  # garanti par assert drift<1e-5 dans la boucle

print(f'\n[✓/✗] std(μ_total) > baseline×{SUCCESS_STD_GAIN:.2f} : {"✓" if OK_STD else "✗"}')
print(f'[✓/✗] RMSE(μ_causal) < baseline×{SUCCESS_RMSE_MAX:.2f} : {"✓" if OK_RMSE else "✗"}')
print(f'[✓/✗] μ_causal_frac ≥ {SUCCESS_FRAC_MIN:.2f}           : {"✓" if OK_FRAC else "✗"}')
print(f'[✓/✗] A_dag stable (drift<1e-5)          : ✓')

if OK_STD and OK_RMSE and OK_FRAC:
    print('\n✓ VERDICT GO — Stage 1 + SRE prêt.')
    print('Prochaines étapes :')
    print(f'  1. Charger {CKPT_SRE_OUT.name} dans le pipeline Stage 2')
    print(f'  2. Mettre sigma_data={sigma_result["sigma_data_new"]:.5f} dans la config EDM Karras')
    print(f'  3. Relancer phase3_mu_HR_probe.ipynb (α* probe) avec le nouveau Stage 1')
    print(f'  4. Si F1@p99 > 0.512 : GO pour réentraîner Stage 2')
else:
    print('\n✗ VERDICT PARTIEL — Diagnostics :')
    if not OK_STD:
        print(f'  std(μ_total) insuffisant ({std_t_f:.4f} vs cible {std_t_b*SUCCESS_STD_GAIN:.4f})')
        print('    → Essayer PHASE_B_EPOCHS=10 ou PHASE_B_LR_SRE=2e-4')
    if not OK_RMSE:
        print(f'  RMSE(μ_causal) dégradé (+{100*(rmse_c_f/rmse_c_b-1):.1f}%)')
        print('    → Augmenter LAMBDA_CAUSAL_END vers 0.5, réduire PHASE_B_LR_BACKBONE')
    if not OK_FRAC:
        print(f'  μ_causal_frac={frac_f:.3f} trop bas (SRE domine)')
        print('    → Ajouter epochs Phase A ou augmenter LAMBDA_CAUSAL_START')

# Sauvegarder le résumé JSON
summary = {
    'baseline': metrics_baseline,
    'after_phase_a': metrics_after_a,
    'final': {k: final_probe[k] for k in ('std_c','std_t','rmse_c','rmse_t','delta_std','frac')},
    'sigma_data': sigma_result,
    'ok_std': OK_STD, 'ok_rmse': OK_RMSE, 'ok_frac': OK_FRAC,
    'verdict': 'GO' if (OK_STD and OK_RMSE and OK_FRAC) else 'PARTIAL',
    'phase_a_losses': phase_a_losses,
    'phase_b_epochs': len(phase_b_metrics),
}
(ORACLE_9N / 'phase5_summary.json').write_text(json.dumps(summary, indent=2))
print(f'\nRésumé sauvegardé → {ORACLE_9N}/phase5_summary.json')
print('=' * 70)